# Decoding: beam search with the char-LM and word LM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/decode.ipynb)

The runtime's pipeline with the T4 standing in for the fabric. The latest trained acoustic model (AM) and character model (CM) run in torch on the GPU, the AM on the int8 features the fabric will get, and their outputs go through the beam search the board runs (`shared/src/shared/decode/`): CTC prefix beam search fused with the CM and OpenSLR's pruned trigram as the word LM (WLM), the paper's N-best search. Tunes the LM weights on a sample of dev-clean, then reports WER and CER on test-clean for each piece added in turn: greedy, beam search alone, + CM, + WLM, + both. **Runtime: T4 GPU** (the AM and CM run on it; the search itself is CPU-bound Python). Saves the log and the results to a run folder on Google Drive.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys
import tarfile
import urllib.request

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "main"  # set to a feature branch to test it before merging

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# kenlm (the word LM) has no wheels, and the binding in its latest sdist (0.3.0) was
# generated by Cython 0.29, which doesn't compile on Python 3.13 (kpu/kenlm#468 regenerates
# it and is unmerged). So build it from the sdist with the binding regenerated by Cython 3,
# before shared's wlm extra asks for it: about a minute.
pip = [sys.executable, "-m", "pip", "install", "-q"]
KENLM = (
    "https://files.pythonhosted.org/packages/02/6e/"
    "f4e159cef39ea4ef2244b4aa32857d447d60a5d0ee269e1b1f8d3723c03e/kenlm-0.3.0.tar.gz"
)
kenlm_dir = "/content/kenlm-0.3.0"
if not os.path.isdir(kenlm_dir):
    with urllib.request.urlopen(KENLM) as f, tarfile.open(fileobj=f, mode="r|gz") as tar:
        tar.extractall("/content", filter="data")
subprocess.run([*pip, "cython>=3.0.11"], check=True)
binding = os.path.join(kenlm_dir, "python", "kenlm")
cython = [sys.executable, "-m", "cython", "--cplus", f"{binding}.pyx", "-o", f"{binding}.cpp"]
subprocess.run(cython, check=True)
subprocess.run([*pip, kenlm_dir], check=True)

# Installs this checkout's packages and their dependencies: shared/ (the vocab and the
# decoder, with kenlm, built above, for the word LM) and training (jiwer, soundfile), in one
# go so training's dependency on shared resolves to this checkout. torch and torchaudio come
# with Colab.
shared_dir = os.path.join(REPO_DIR, "shared")
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run(
    [*pip, "-e", f"{shared_dir}[wlm]", "-e", training_dir],
    check=True,
)

# The editable installs only take effect at interpreter start, so put both src/ dirs on
# the path for this already-running kernel.
for d in (shared_dir, training_dir):
    sys.path.insert(0, os.path.join(d, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

`AM_RUN` and `CM_RUN` default to the newest run folders `am_training.ipynb` and `cm_training.ipynb` saved to, in `MyDrive/enph479-stt/runs/` of the mounted account; set either to a run folder's path to decode with another. `SUFFIX = "_6bit"` decodes with the 6-bit models, the ones the fabric runs; `""` uses the float ones. The results go to `MyDrive/enph479-stt/runs/<start time>-decode/`. The downloaded archives are kept in `MyDrive/enph479-stt/data/`, the cache the training notebooks use.

The LM weights are tuned on `TUNE_UTTS` random dev-clean utterances with a narrower beam (`TUNE_BEAM`), one decode per grid point, and the results are reported on `EVAL_UTTS` random test-clean utterances (`None` for all 2,620) at the full beam, the paper's 128. Not yet timed on Colab: the decode prints seconds per utterance every 200 utterances, so shrink the grid or the sizes if it runs long.

In [ ]:
import glob
import time

from google.colab import drive

from shared.decode.beam import Config

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/enph479-stt/runs"


def latest_run(checkpoint: str) -> str:
    """The newest run folder holding ``checkpoint`` (run folders are named by start time)."""
    return os.path.dirname(max(glob.glob(f"{RUNS}/*/{checkpoint}")))


AM_RUN = latest_run("am.pt")  # or the path of another AM run
CM_RUN = latest_run("cm.pt")  # or the path of another CM run
SUFFIX = "_6bit"
RUN_DIR = f"{RUNS}/{time.strftime('%Y%m%d-%H%M')}-decode"
DATA_CACHE = "/content/drive/MyDrive/enph479-stt/data"

TUNE_UTTS = 100
EVAL_UTTS = 500
TUNE_BEAM = 32
GRID = {"alpha": [0.25, 0.5, 0.75], "beta": [0.1, 0.25, 0.5], "gamma": [2.0, 3.0, 4.0]}
cfg = Config(beam=128)
print(f"AM {AM_RUN}")
print(f"CM {CM_RUN}")
print(f"saving to {RUN_DIR}")
cfg

## 3. Models and data

Loads both checkpoints (the AM's carries its CMVN statistics), puts dev-clean (337 MB), test-clean (346 MB) and the word LM (34 MB gzipped ARPA: 200K words, 2.5M bigrams, 1.1M trigrams, from the same LM text as the CM with the dev and test books excluded) in `/content/data` (copied from `DATA_CACHE` on Drive, or downloaded from OpenSLR and cached there the first time), and picks the tuning utterances from dev-clean and the evaluation ones from test-clean. The word LM's perplexity and OOV rate on the test-clean transcripts are its own sanity check, before any decoding.

In [ ]:
import random

from shared.decode.wlm import KenLM, perplexity
from training.am.dataset import list_utterances
from training.data.librispeech import download_subset, download_word_lm
from training.decode import run

am, cm_model, mean, std = run.load_models(
    f"{AM_RUN}/am{SUFFIX}.pt", f"{CM_RUN}/cm{SUFFIX}.pt", "cuda"
)
cm = run.TorchCharLM(cm_model)
dev = list_utterances(download_subset("dev-clean", "/content/data", DATA_CACHE))
test = list_utterances(download_subset("test-clean", "/content/data", DATA_CACHE))
wlm = KenLM(download_word_lm("/content/data", DATA_CACHE))

rng = random.Random(0)
tune_idx = rng.sample(range(len(dev)), TUNE_UTTS)
eval_items = test if EVAL_UTTS is None else rng.sample(test, EVAL_UTTS)
dev_refs = [text for _, text in dev]
eval_refs = [text for _, text in eval_items]
print(f"tune on {len(tune_idx)} of dev-clean's {len(dev)} utts")
print(f"evaluate on {len(eval_items)} of test-clean's {len(test)}")

ppl, oov = perplexity(wlm, [text for _, text in test])
run.log("decode.log", f"word LM on test-clean: perplexity {ppl:.0f}, OOV rate {oov:.2%}")

## 4. AM output

Runs the AM once over all of dev-clean and once over the evaluation utterances, and keeps the per-frame log-probs, so every decode below reuses them. The features go in as the int8 the fabric takes (CMVN, then x 2^5, rounded and clamped), not the float ones training uses. Greedy decoding (per-frame argmax, CTC collapse) on all of dev-clean is the training notebook's last test CER on int8 input: the difference between the two is what the int8 features cost.

In [ ]:
dev_logps = run.am_log_probs(am, dev, mean, std)
wer, cer = run.error_rates(dev_refs, run.greedy(dev_logps))
run.log("decode.log", f"greedy, all of dev-clean: WER {wer:.4f} CER {cer:.4f}")
eval_logps = run.am_log_probs(am, eval_items, mean, std)

## 5. Tune the LM weights

Grid search over `alpha` (CM weight), `beta` (WLM weight) and `gamma` (bonus per word) with both LMs, at `TUNE_BEAM`, for the lowest WER on the tuning utterances. Every point is logged.

The grid is centred on the first decode's best (`runs/20261102-2019-decode`: alpha 0.5, beta 0.5, gamma 2.0), which sat on the edge of that run's grid in beta and gamma. If the best point lands on an edge again, move the grid that way and rerun.

In [ ]:
from dataclasses import replace

best = run.tune(
    [dev_logps[i] for i in tune_idx],
    [dev_refs[i] for i in tune_idx],
    replace(cfg, beam=TUNE_BEAM),
    GRID,
    "decode.log",
    cm,
    wlm,
)
cfg = replace(best, beam=cfg.beam)
cfg

## 6. Evaluate

Each piece added in turn, on the test-clean utterances at the full beam, with the tuned weights (the rows with one LM keep that LM's weight from the joint tuning). Greedy to "beam, no LM" is what summing over alignments buys; the rest is what each LM buys. "Beam, no LM" runs with `gamma = 0`: the word bonus is there to offset the LMs' preference for short output, and with no LM it only pays the search to insert spaces.

In [ ]:
systems = {
    "greedy": lambda: run.greedy(eval_logps),
    "beam, no LM": lambda: run.decode(eval_logps, replace(cfg, gamma=0.0)),
    "beam + CM": lambda: run.decode(eval_logps, cfg, cm),
    "beam + WLM": lambda: run.decode(eval_logps, cfg, wlm=wlm),
    "beam + CM + WLM": lambda: run.decode(eval_logps, cfg, cm, wlm),
}
results = {}
for name, decode in systems.items():
    t0 = time.perf_counter()
    wer, cer = run.error_rates(eval_refs, decode())
    results[name] = {"wer": wer, "cer": cer}
    secs = time.perf_counter() - t0
    run.log("decode.log", f"{name}: WER {wer:.4f} CER {cer:.4f} ({secs:.0f}s)")

## 7. Save to Drive

Writes the log and `decode.json` (which runs and weights were decoded with, the tuned config and the results table) to the run folder, then flushes Drive.

In [ ]:
import json
import shutil
from dataclasses import asdict
from pathlib import Path

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
shutil.copy("decode.log", RUN_DIR)
summary = {
    "am": f"{AM_RUN}/am{SUFFIX}.pt",
    "cm": f"{CM_RUN}/cm{SUFFIX}.pt",
    "tune_utts": f"{len(tune_idx)} of dev-clean",
    "eval_utts": f"{len(eval_items)} of test-clean",
    "config": asdict(cfg),
    "results": results,
}
Path(RUN_DIR, "decode.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"saved the log and decode.json to {RUN_DIR}")
drive.flush_and_unmount()